In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()


# Model class
class Student(BaseModel):
    id: int
    name: str
    age: int


students = []


# GET - Get all students
@app.get("/students")
def get_students():
    return students


# GET - Get one student
@app.get("/students/{student_id}")
def get_student(student_id: int):
    for student in students:
        if student.id == student_id:
            return student

    return {"message": "Student not found"}


# POST - Add student
@app.post("/students")
def add_student(student: Student):
    students.append(student)
    return {"message": "Student added", "student": student}


# PUT - Update complete student
@app.put("/students/{student_id}")
def update_student(student_id: int, student_data: Student):
    for student in students:
        if student.id == student_id:
            student.name = student_data.name
            student.age = student_data.age

            return {"message": "Student updated", "student": student}

    return {"message": "Student not found"}


# DELETE - Delete student
@app.delete("/students/{student_id}")
def delete_student(student_id: int):
    for student in students:
        if student.id == student_id:
            students.remove(student)
            return {"message": "Student deleted"}

    return {"message": "Student not found"}

In [ ]:
from fastapi import FastAPI
import mysql.connector

app = FastAPI()


# MySQL connection
def get_connection():
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="your_password",
        database="student_db"
    )


# Class for student data
class Student:
    def __init__(self, name, age):
        self.name = name
        self.age = age


# GET - Get all students
@app.get("/students")
def get_students():

    db = get_connection()
    cursor = db.cursor(dictionary=True)

    cursor.execute("SELECT * FROM students")
    students = cursor.fetchall()

    cursor.close()
    db.close()

    return students


# GET - Get one student
@app.get("/students/{student_id}")
def get_student(student_id: int):

    db = get_connection()
    cursor = db.cursor(dictionary=True)

    cursor.execute(
        "SELECT * FROM students WHERE id = %s",
        (student_id,)
    )

    student = cursor.fetchone()

    cursor.close()
    db.close()

    return student


# POST - Add student
@app.post("/students")
def add_student(name: str, age: int):

    db = get_connection()
    cursor = db.cursor()

    cursor.execute(
        "INSERT INTO students (name, age) VALUES (%s, %s)",
        (name, age)
    )

    db.commit()

    cursor.close()
    db.close()

    return {"message": "Student added successfully"}


# PUT - Update student
@app.put("/students/{student_id}")
def update_student(student_id: int, name: str, age: int):

    db = get_connection()
    cursor = db.cursor()

    cursor.execute(
        "UPDATE students SET name = %s, age = %s WHERE id = %s",
        (name, age, student_id)
    )

    db.commit()

    cursor.close()
    db.close()

    return {"message": "Student updated successfully"}


# DELETE - Delete student
@app.delete("/students/{student_id}")
def delete_student(student_id: int):

    db = get_connection()
    cursor = db.cursor()

    cursor.execute(
        "DELETE FROM students WHERE id = %s",
        (student_id,)
    )

    db.commit()

    cursor.close()
    db.close()

    return {"message": "Student deleted successfully"}